# Proyecto LOST — MountainCarContinuous-v0 con Q-Learning

Notebook principal del Proyecto LOST. Acá hacemos:

1. Setup del ambiente y verificación de los espacios.
2. Demo del `Discretizer`.
3. Smoke test del `QLearningAgent` (entrenamiento corto + curva de aprendizaje).
4. Test de persistencia (`save` / `load`).
5. (Más adelante) Grid search de hiperparámetros y comparación contra Dyna-Q.

El razonamiento y la justificación están en `Documentacion.md` (en la raíz del repo).

In [ ]:
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt

from discretization import Discretizer
from q_learning_agent import QLearningAgent

## 1. Setup del ambiente

Usamos `render_mode=None` durante entrenamiento (mucho más rápido que `human` o `rgb_array`). Solo activamos render al final para la demo visual.

In [ ]:
env_id = 'MountainCarContinuous-v0'
env = gym.make(env_id)

print('Observation space:', env.observation_space)
print('Action space:     ', env.action_space)

## 2. Discretización

Convertimos `(x, v)` continuos a un par de índices `(xi, vi)` y la fuerza continua `a ∈ [-1, 1]` a uno de `n_actions` valores discretos.

In [ ]:
disc = Discretizer(n_bins_x=40, n_bins_v=40, n_actions=5)
print('q_shape:', disc.q_shape)
print('acciones:', disc.actions)

# Ejemplo: observación → estado
obs = np.array([-0.4, 0.02])
print(f'obs={obs} → state={disc.get_state(obs)}')

## 3. Smoke test del Q-Learning agent

Configuración media (40×40 bins, 5 acciones) + reward shaping potential-based con `coef=300`. 500 episodios para validar la pipeline rápido.

In [ ]:
agent = QLearningAgent(
    discretizer=disc,
    alpha=0.1,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.995,
    optimistic_init=0.0,
    reward_shaping=True,
    shaping_coef=300.0,
    seed=42,
)

history = agent.train_agent(env, episodes=500, verbose_every=100, env_seed=42)

### Curva de aprendizaje

Reward por episodio (sin shaping, así es comparable entre configs) y tasa de éxito en ventana móvil.

In [ ]:
def plot_history(history, title):
    rewards = np.array(history['rewards'])
    successes = np.array(history['success'])
    window = 50
    moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
    success_rate = np.convolve(successes, np.ones(window)/window, mode='valid')

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    axes[0].plot(rewards, alpha=0.3, label='reward/ep')
    axes[0].plot(np.arange(len(moving_avg)) + window - 1, moving_avg, color='C1', label=f'media móvil ({window})')
    axes[0].set_ylabel('Reward acumulado'); axes[0].grid(alpha=0.3); axes[0].legend(); axes[0].set_title(title)
    axes[1].plot(np.arange(len(success_rate)) + window - 1, success_rate, color='C2')
    axes[1].set_ylabel(f'Tasa éxito (ventana {window})'); axes[1].set_xlabel('Episodio'); axes[1].set_ylim(-0.05, 1.05); axes[1].grid(alpha=0.3)
    fig.tight_layout()
    return fig

fig = plot_history(history, 'Smoke test — Q-Learning + potential-based shaping')
plt.show()

### Test greedy (política sin exploración)

In [ ]:
metrics = agent.test_agent(env, episodes=10)
print(f"avg_reward   = {metrics['avg_reward']:7.2f}")
print(f"success_rate = {metrics['success_rate']:.2%}")
print(f"avg_steps    = {metrics['avg_steps']:.1f}")

## 4. Persistencia

`agent.save(path)` guarda Q + config del discretizer + hiperparámetros en un `.pkl`. `QLearningAgent.load(path)` reconstruye el agente sin tener que recordar la configuración. **Esto es obligatorio para la entrega** (la consigna lo marca explícitamente).

In [ ]:
agent.save('models/smoke_test.pkl')

loaded = QLearningAgent.load('models/smoke_test.pkl')
loaded_metrics = loaded.test_agent(env, episodes=5)
print('Modelo cargado, métricas:', loaded_metrics)

## 5. Grid search y Dyna-Q

*(Pendiente — se agregan en los próximos pasos.)*

- Grid search sobre `(bins, alpha, gamma, epsilon_decay, shaping)` para Q-Learning.
- Implementación de `DynaQAgent` (Sutton & Barto cap. 8.1–8.2) y comparación contra Q-Learning.

In [ ]:
env.close()